In [14]:
# Tv
# Thermostat
# Lights
# Washing Machine
# Dishwasher
# Dryer
# Electric Vehicle Charger

In [15]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpBinary, LpContinuous, LpStatus, PULP_CBC_CMD
import json

def run_optimization_hybrid(appliances, prices, user_preferences):

    T = len(prices)
    availability = user_preferences.get("availability", [])
    temp_home = user_preferences.get("thermostat_temp_home", 72)
    temp_away = user_preferences.get("thermostat_temp_away", 78)
    hvac_lead = user_preferences.get("hvac_lead_time", 1)

    prob = LpProblem("energy_optimization", LpMinimize)

    x = {}          # appliance run vars
    start_vars = {} # start-time vars for block modeling
    temp = {}       # HVAC temp vars

    # =========================
    # Variables
    # =========================
    for i, app in enumerate(appliances):
        if app["type"] in ["cycle", "intermittent"]:
            for t in range(T):
                x[i, t] = LpVariable(f"x_{i}_{t}", cat=LpBinary)

        if app["type"] == "cycle":
            D = app["duration"]
            start_vars[i] = [LpVariable(f"start_{i}_{t}", cat=LpBinary) for t in range(T - D + 1)]

        elif app["type"] == "hvac":
            for t in range(T):
                temp[i, t] = LpVariable(f"temp_{i}_{t}", lowBound=60, upBound=85, cat=LpContinuous)

    # =========================
    # Objective: Minimize Cost
    # =========================
    prob += lpSum(
        x[i, t] * app.get("power", 0) * prices[t]
        for i, app in enumerate(appliances)
        if app["type"] in ["cycle", "intermittent"]
        for t in range(T)
    ) + lpSum(
        app.get("power", 0) * prices[t]
        for i, app in enumerate(appliances)
        if app["type"] == "hvac"
        for t in range(T)
    )

    # =========================
    # Constraints
    # =========================
    for i, app in enumerate(appliances):

        # ---------------------
        # Cycle appliances: block modeling
        # ---------------------
        if app["type"] == "cycle":
            D = app["duration"]
            start = start_vars[i]

            # Exactly one start time
            prob += lpSum(start) == 1

            # Link start -> run hours (consecutive block)
            for t in range(T):
                possible_starts = [start[s] for s in range(max(0, t - D + 1), min(t + 1, T - D + 1))]
                if possible_starts:
                    prob += lpSum(possible_starts) == x[i, t]

            # Respect availability for non-smart cycles
            if not app.get("smart_enabled", True):
                for t in range(T):
                    if t not in availability:
                        prob += x[i, t] == 0

        # ---------------------
        # Intermittent appliances
        # ---------------------
        elif app["type"] == "intermittent":
            prob += lpSum(x[i, t] for t in range(T)) <= app.get("max_hours", T)
            for t in range(T):
                if t not in availability:
                    prob += x[i, t] == 0

        # ---------------------
        # HVAC
        # ---------------------
        elif app["type"] == "hvac":
            return_times = []
            for t in availability:
                if t - 1 >= 0 and (t - 1) not in availability:
                    return_times.append(t)
            pre_cool_hours = [rt - hvac_lead for rt in return_times if rt - hvac_lead >= 0]

            for t in range(T):
                if t in availability or t in pre_cool_hours:
                    prob += temp[i, t] == temp_home
                else:
                    prob += temp[i, t] == temp_away

    # ---------------------
    # EV Charger must finish before departure
    # ---------------------
    for i, app in enumerate(appliances):
        if app["name"] == "EV Charger":
            departure = app.get("departure_hour", T)
            for t in range(departure, T):
                prob += x[i, t] == 0

    # ---------------------
    # Washer -> Dryer sequencing
    # ---------------------
    washer_idx = next((i for i, a in enumerate(appliances) if a["name"] == "Washing Machine"), None)
    dryer_idx = next((i for i, a in enumerate(appliances) if a["name"] == "Dryer"), None)

    if washer_idx is not None and dryer_idx is not None:
        D_w = appliances[washer_idx]["duration"]
        D_d = appliances[dryer_idx]["duration"]
        start_w = start_vars[washer_idx]
        start_d = start_vars[dryer_idx]

        # Dryer must start immediately after washer
        for t_w in range(len(start_w)):
            t_d = t_w + D_w
            if t_d < len(start_d):
                prob += start_d[t_d] >= start_w[t_w]
            else:
                # Prevent washer starting too late that dryer can't fit
                prob += start_w[t_w] == 0

        # No overlap in run hours
        for t in range(T):
            prob += x[washer_idx, t] + x[dryer_idx, t] <= 1

    # =========================
    # Solve
    # =========================
    prob.solve(PULP_CBC_CMD(msg=0))

    status_str = LpStatus[prob.status]
    if status_str != "Optimal":
        return {"status": status_str}

    # =========================
    # Minimal output
    # =========================
    schedule_output = []
    total_cost = 0

    for i, app in enumerate(appliances):
        if app["type"] in ["cycle", "intermittent"]:
            run_times = [t for t in range(T) if (x[i, t].value() or 0) >= 0.99]
            total_cost += sum(prices[t] * app.get("power", 0) for t in run_times)
            schedule_output.append({
                "appliance": app["name"],
                "run_times": run_times
            })
        elif app["type"] == "hvac":
            total_cost += sum(prices[t] * app.get("power", 0) for t in range(T))
            schedule_output.append({
                "appliance": app["name"],
                "run_times": "continuous"
            })

    return {
        "status": "OPTIMAL",
        "schedule": schedule_output,
        "total_estimated_cost": round(total_cost, 2)
    }


In [16]:
import json
#from optimization import run_optimization_hybrid

# Example: loading from a JSON string (could be request.body)
with open('sample.json', 'r') as f:
  json_input = f.read()

data = json.loads(json_input)

user_preferences = data['user_preferences']
appliances = data['appliances']
prices = data['prices']

# Run optimization
result = run_optimization_hybrid(appliances, prices, user_preferences)

# Print output
print("Optimal Schedule:")
for a in result["schedule"]:
    print(f"{a['appliance']}: {a['run_times']}")
print(f"Estimated Total Cost: ${result['total_estimated_cost']}")

Optimal Schedule:
TV: []
Lights: []
Washing Machine: [3]
Dishwasher: [4]
Dryer: [4, 5]
EV Charger: [2, 3, 4, 5]
Thermostat: continuous
Estimated Total Cost: $19.05
